# PubMedBERT ProteinStructure — BioNER on Google Colab

**Model:** `PDBEurope/BiomedNLP-PubMedBERT-ProteinStructure-NER-v3.1`  ·  **Family:** Transformer (fixed)

PubMedBERT for protein-structure entities (residue/site/domain/complex).

This model runs fine on CPU, but a GPU speeds it up. Set **Runtime → Change runtime type → T4 GPU** before running.

This notebook reuses the shared harness (39-type schema, 69-sentence gold corpus,
evaluation metrics) from the `ner/` package in the repo. It:
1. checks the GPU, 2. installs this model's backend, 3. clones the repo,
4. runs a sanity check, 5. benchmarks on the gold corpus, 6. plots results.

## 1 · Environment / GPU check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — Runtime -> Change runtime type -> T4 GPU")

## 2 · Install this model's backend

In [ ]:
!pip install -q -U transformers

## 3 · Get the shared harness

Clones this repo (branch `ner_models`) so we can import `ner.models`,
`ner.corpus`, and `ner.evaluation`. The `ner/` folder must be committed to the
branch. If you haven't pushed it yet, run `git add ner && git commit && git push`
locally first, or upload the `ner/` folder to `/content` manually.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/ermijeremy/bio-semantic-parser"
BRANCH   = "ner_models"
CLONE_DIR = "/content/bio-semantic-parser"

if not os.path.isdir(CLONE_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
        check=True,
    )

if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)

assert os.path.isdir(os.path.join(CLONE_DIR, "ner")), (
    "ner/ package not found in the clone — commit & push the ner/ folder to the "
    f"'{BRANCH}' branch, or upload it to {CLONE_DIR} manually."
)
print("Harness ready:", CLONE_DIR)

## 4 · Load the model & sanity check

In [ ]:
from ner.models import load_model
from ner.schema import tier_of

KEY = "pubmedbert_protein_structure"
model = load_model(KEY)          # lazy: downloads + loads on first predict
print("Loaded:", model.name, "|", model.model_id)

text = 'The p53 protein is phosphorylated at Ser15 within its DNA-binding domain.'
print("\nSanity check:", text)
for e in sorted(model.predict(text), key=lambda x: x.start):
    print(f"  [{e.label} / {tier_of(e.label)}] {e.text!r}  ({e.score:.2f})")

## 5 · Benchmark on the gold corpus

Exact & partial (overlap + same-type) precision/recall/F1 over all 69 sentences,
plus tier-1 F1, schema coverage, latency, and GPU peak memory.

In [ ]:
from ner.corpus import CORPUS, corpus_stats
from ner.evaluation import evaluate_model

print(corpus_stats()["n_sentences"], "sentences,",
      corpus_stats()["total_gold_entities"], "gold entities\n")

report = evaluate_model(model.name, model.predict, CORPUS)

o, t1 = report["overall"], report["tier1_only"]
print(f"Overall   partial F1: {o['partial_f1']}   exact F1: {o['exact_f1']}")
print(f"          precision:  {o['partial_precision']}   recall: {o['partial_recall']}")
print(f"Tier-1    partial F1: {t1['partial_f1']}")
print(f"Coverage: {report['type_coverage_pct']}% of schema types")
print(f"Latency:  avg {report['avg_latency_ms']} ms   p95 {report['p95_latency_ms']} ms")
print(f"GPU peak: {report['gpu_peak_mb']} MB")
if report["errors"]:
    print(f"\n{len(report['errors'])} sentence error(s); first:", report["errors"][0])

## 6 · Per-type F1 & summary plot

In [ ]:
import matplotlib.pyplot as plt

per_type = {t: m["partial_f1"] for t, m in report["per_type"].items()}
per_type = dict(sorted(per_type.items(), key=lambda kv: kv[1], reverse=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, max(4, len(per_type) * 0.28)),
                               gridspec_kw={"width_ratios": [2, 1]})

ax1.barh(list(per_type.keys())[::-1], list(per_type.values())[::-1], color="#3B9ED8")
ax1.set_xlim(0, 1); ax1.set_xlabel("partial F1")
ax1.set_title(f"{model.name} — per-type partial F1")
ax1.grid(axis="x", alpha=0.3)

summary = {
    "Partial F1": report["overall"]["partial_f1"],
    "Tier-1 F1": report["tier1_only"]["partial_f1"],
    "Precision": report["overall"]["partial_precision"],
    "Recall": report["overall"]["partial_recall"],
    "Coverage": report["type_coverage_pct"] / 100,
}
ax2.bar(range(len(summary)), list(summary.values()),
        color=["#185FA5", "#3B9ED8", "#639922", "#EF9F27", "#8B2FC9"])
ax2.set_xticks(range(len(summary))); ax2.set_xticklabels(list(summary.keys()), rotation=35, ha="right")
ax2.set_ylim(0, 1); ax2.set_title("Summary"); ax2.grid(axis="y", alpha=0.3)
for i, v in enumerate(summary.values()):
    ax2.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"ner_report_{KEY}.png", dpi=140, bbox_inches="tight")
plt.show()

## 7 · Save the JSON report

In [ ]:
import json

with open(f"ner_report_{KEY}.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved:", f"ner_report_{KEY}.json", "and", f"ner_report_{KEY}.png")